### Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

!pip install -q snntorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import snntorch as snn
from snntorch import surrogate

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Configuration

In [ ]:
SEED = 42
WINDOW = 24
HORIZON = 1

BATCH_SIZE = 128
EPOCHS = 30
LEARNING_RATE = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

### Load Dataset

In [ ]:
df = pd.read_csv("data/House_2.csv")

df["Time"] = pd.to_datetime(df["Time"])
df = df.sort_values("Time")

# Select all relevant columns: Time, Aggregate, and Appliance1-9
cols_to_keep = ["Time", "Aggregate"] + [f"Appliance{i}" for i in range(1, 10)]
df = df[cols_to_keep].dropna()

# Resample all columns to 15-minute intervals
df = df.set_index("Time").resample("15min").mean().dropna().reset_index()

print(df.shape)
print(df.columns.tolist())
print(df.head())

In [ ]:
df["Target"] = df["Aggregate"].shift(-HORIZON)
df = df.dropna().reset_index(drop=True)

print(df.head())
print(df.shape)

In [ ]:

# Add hour features
df["hour"] = df["Time"].dt.hour
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

print("Columns:", df.columns.tolist())
print(df.head())


### Preprocessing

In [ ]:
# Chronological split
n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

In [ ]:
# 12 input features: Aggregate + Appliance1-9 + hour_sin + hour_cos
feature_cols = ["Aggregate"] + [f"Appliance{i}" for i in range(1, 10)] + ["hour_sin", "hour_cos"]

scaler = StandardScaler()

train_scaled = scaler.fit_transform(train_df[feature_cols])
val_scaled = scaler.transform(val_df[feature_cols])
test_scaled = scaler.transform(test_df[feature_cols])

print(f"Features ({len(feature_cols)}): {feature_cols}")
print("Train mean:", train_scaled.mean(axis=0))
print("Train std:", train_scaled.std(axis=0))
print("Scaled shape:", train_scaled.shape)

In [ ]:
def create_sequences(data, window=WINDOW):
    X = []
    y = []

    for i in range(len(data) - window):
        X.append(data[i:i + window])
        y.append(data[i + window])

    return np.array(X), np.array(y)


X_train, y_train = create_sequences(train_scaled)
X_val, y_val = create_sequences(val_scaled)
X_test, y_test = create_sequences(test_scaled)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

print(X_train.shape, y_train.shape)

In [ ]:
train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

### RSNN Model

In [ ]:
class RSNNForecaster(nn.Module):
    def __init__(self, input_size=12):
        super().__init__()

        self.fc_in = nn.Linear(input_size, 32)
        self.rec = nn.Linear(32, 32)
        self.lif = snn.Leaky(beta=0.9, spike_grad=surrogate.fast_sigmoid())
        self.output = nn.Linear(32, 1)

    def forward(self, x):
        # x: [batch, time, feature]
        batch_size = x.size(0)
        mem = self.lif.init_leaky()
        hidden = torch.zeros(batch_size, 32, device=x.device)

        for t in range(x.size(1)):
            cur = self.fc_in(x[:, t, :])
            rec = self.rec(hidden)
            combined = cur + rec
            spk, mem = self.lif(combined, mem)
            hidden = spk

        return self.output(hidden)


model = RSNNForecaster(input_size=12).to(DEVICE)
print(model)

In [ ]:
criterion = nn.HuberLoss(delta=1.0)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

### Training Loop

In [ ]:
train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)
    train_losses.append(train_loss)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            val_loss += loss.item()

    val_loss /= len(val_loader)
    val_losses.append(val_loss)

    print(f"Epoch {epoch + 1:02d}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Huber Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

### Evaluation

In [ ]:
model.eval()

predictions = []
actuals = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        pred = model(X_batch)

        predictions.append(pred.cpu().numpy())
        actuals.append(y_batch.numpy())

predictions = np.concatenate(predictions).flatten()
actuals = np.concatenate(actuals).flatten()

# Inverse transform: create dummy arrays with 12 features
# Put predictions/actuals in first column (Aggregate), zeros elsewhere
n_samples = len(predictions)
predictions_full = np.zeros((n_samples, len(feature_cols)))
actuals_full = np.zeros((n_samples, len(feature_cols)))

predictions_full[:, 0] = predictions
actuals_full[:, 0] = actuals

predictions_watts = scaler.inverse_transform(predictions_full)[:, 0]
actuals_watts = scaler.inverse_transform(actuals_full)[:, 0]

mae = mean_absolute_error(actuals_watts, predictions_watts)
rmse = np.sqrt(mean_squared_error(actuals_watts, predictions_watts))
r2 = r2_score(actuals_watts, predictions_watts)

print(f"MAE:  {mae:.2f} W")
print(f"RMSE: {rmse:.2f} W")
print(f"R²:   {r2:.4f}")

In [ ]:
points = 96

plt.figure(figsize=(12, 5))
plt.plot(actuals_watts[:points], label="Actual")
plt.plot(predictions_watts[:points], label="Predicted")
plt.xlabel("15-minute interval")
plt.ylabel("Power (W)")
plt.title("RSNN: Actual vs Predicted Power")
plt.legend()
plt.grid(True)
plt.show()